# Bachelor Thesis

© 2026 Yvan Richard   
University of St. Gallen, Spring Term 2026

## Final Sample Construction

---
## Foreword

In this notebook, my goal is to build the final main sample for the models developed in my thesis.


## 1. Libraries & Data

I first load the relevant libraries and data.

In [1]:
# libs
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [2]:
# data
main = pd.read_csv("../../../data/processed/main_with_svi.csv")

/var/folders/7v/_v_y1jpx0rl056gg5rkjsw4r0000gn/T/ipykernel_4467/1285260102.py:2: DtypeWarning: Columns (11) have mixed types. Specify dtype option on import or set low_memory=False.
  main = pd.read_csv("../../../data/processed/main_with_svi.csv")


## 2. Missing Values & EDA

In this section, the goal is to identify the missing values that will be an issue for my analysis. The main constraint I will face is the serious missingness in Google Trends search volume index (SVI).

In [3]:
# parse dates
main['date'] = pd.to_datetime(main['date'])

# rename TSSVI to svi
main.rename(columns={'TSSVI': 'svi'}, inplace=True)

### 2.1. Missing Google Trends Data

I study the missingness of Google Trends data.

In [7]:
# number of unique tickers with non-nan svi
print(f"Number of unique tickers with non-nan svi: {main[main['svi'].notna()]['ticker'].nunique()}")

# total number of unique tickers
print(f"Total number of unique tickers: {main['ticker'].nunique()}")

# drop rows with missing values in svi
main = main[main['svi'].notna()]

Number of unique tickers with non-nan svi: 1698
Total number of unique tickers: 9007


#### herding events

I also study the influence on the number of herding events.

In [8]:
# sort values by ticker and date
rh_herd = main.copy()  # create a copy to avoid SettingWithCopyWarning
rh_herd = rh_herd.sort_values(["ticker", "date"]).copy()

# userratio = users_close(t) / users_close(t-1)
rh_herd["users_close_lag1"] = rh_herd.groupby("ticker")["users_close"].shift(1)
rh_herd["userratio"] = rh_herd["users_close"] / rh_herd["users_close_lag1"]

# flag np.inf and -np.inf values in userratio as NaN (can happen when users_close_lag1 is zero)
rh_herd["userratio"].replace([np.inf, -np.inf], np.nan, inplace=True)

# shortlist the possible herding events based on the criteria of Barbet et al. (2022)
rh_herd["shortlist_herd"] = (
    (rh_herd["userratio"] > 1)
    & (rh_herd["users_close_lag1"] >= 100)
    & rh_herd["userratio"].notna()
)

# Identify the top 0.5% of shortlisted stocks based on userratio for each day
# Only shortlisted stocks can be flagged as herding events.
rh_herd["rh_herd"] = False
rh_herd.loc[rh_herd["shortlist_herd"], "rh_herd"] = (
    rh_herd.loc[rh_herd["shortlist_herd"]]
    .groupby("date")["userratio"]
    .transform(lambda x: x >= x.quantile(0.995))
)

# Report on the number of herding events identified, number of
# unique tickers involved
num_herding_events = rh_herd["rh_herd"].sum()
num_unique_tickers = rh_herd.loc[rh_herd["rh_herd"], "ticker"].nunique()
print(f"Number of herding events identified: {num_herding_events}")
print(f"Number of unique tickers involved: {num_unique_tickers}")

Number of herding events identified: 1324
Number of unique tickers involved: 626


/var/folders/7v/_v_y1jpx0rl056gg5rkjsw4r0000gn/T/ipykernel_4467/2926430459.py:10: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  rh_herd["userratio"].replace([np.inf, -np.inf], np.nan, inplace=True)


This means that I still have a meaningful number of herding events.

#### summary statistics

I compute the summary statistics to compare it to my previous sample.

In [9]:
# users_close_l1: lagged users_close by 1 day
main["users_close_l1"] = main.groupby("ticker")["users_close"].shift(1)

# userchg: change in users from day t-1 to day t
main["userchg"] = main["users_close"] - main["users_close_l1"]

# userratio: users_close(t) / users_close(t-1)
main["userratio"] = main["users_close"] / main["users_close_l1"]

# replace inf and -inf in userratio with NaN
main["userratio"] = main["userratio"].replace([np.inf, -np.inf], np.nan)

# make sure that I take the absolute value of the price
main["prc"] = main["prc"].abs()

# size is defined as prc * shrout
main["size"] = main["prc"] * main["shrout"] / 1e6  # in millions

# build 'daily_buys'
main["daily_buys"] = main["buy_num_trades_retail"]

# build 'daily_sells'
main["daily_sells"] = main["sell_num_trades_retail"]

# build 'net_buys'
main["net_buys"] = main["daily_buys"] - main["daily_sells"]

# build 'taq_retimb'
main['taq_retimb'] = main['net_buys'] / (main['daily_buys'] + main['daily_sells'])

# build binary variable 'has_users' indicating whether there are any users for that ticker on that day
main["has_users"] = (main["users_close"] > 0).astype(int)

In [11]:
# summary stats Panel A
summary_stats_panel_a = main[["users_close", "users_last", "userchg", "userratio", "prc", "size", "ret",
                                "daily_buys", "daily_sells", "net_buys", "taq_retimb", "svi"]].describe().T
print(summary_stats_panel_a)

                count         mean           std           min        25%  \
users_close  751599.0  1751.538171  12042.410379      0.000000  71.000000   
users_last   752163.0  1751.820381  12046.853648      0.000000  71.000000   
userchg      746603.0     8.048003    224.837627 -17015.000000  -1.000000   
userratio    744648.0     1.006496      0.328489      0.000000   0.995244   
prc          871692.0    52.710940    146.122236      0.030000  12.040000   
size         871692.0     2.842404      5.583496      0.000092   0.433359   
ret          871692.0     0.000233      0.039668     -1.000000  -0.012486   
daily_buys   864489.0   145.834746    992.949661      0.000000  16.000000   
daily_sells  864489.0   135.256228    787.351464      0.000000  16.000000   
net_buys     864489.0    10.578519    315.767768 -24007.000000  -9.000000   
taq_retimb   864489.0    -0.000713      0.264821     -1.000000  -0.126437   
svi          871692.0    16.697492     28.789691      0.000000   0.000000   

## 3. Building the Final Sample

For the final sample, I will keep only the Russell 3000 stocks used in the google trends data set. This means that the scope of my analysis will be more restricted than Barber et al. (2022).

In [29]:
# read data
df_final = pd.read_csv("../../../data/processed/main_with_svi.csv")

# parse dates
df_final['date'] = pd.to_datetime(df_final['date'])

# rename TSSVI to svi
df_final.rename(columns={'TSSVI': 'svi'}, inplace=True)

/var/folders/7v/_v_y1jpx0rl056gg5rkjsw4r0000gn/T/ipykernel_4467/571122779.py:2: DtypeWarning: Columns (11) have mixed types. Specify dtype option on import or set low_memory=False.
  df_final = pd.read_csv("../../../data/processed/main_with_svi.csv")


I download CSV data for MSFT, AAPL, ACB, F, and GE because they were the previous most important stocks.

In [30]:
# small pipeline to add selected tickers to the final sample
# read raw data
files = ["../../../data/raw/top_ticks/top_tick_18.csv",
         "../../../data/raw/top_ticks/top_tick_18_2.csv",
         "../../../data/raw/top_ticks/top_tick_19.csv",
         "../../../data/raw/top_ticks/top_tick_19_2.csv",
         "../../../data/raw/top_ticks/top_tick_20.csv",
         "../../../data/raw/top_ticks/top_tick_20_2.csv"]

Once I set the data paths, I load them and process them.

In [31]:
add_df = pd.DataFrame()

for file in files:
    df = pd.read_csv(file)

    # 1) Rename and parse dates
    df = df.rename(columns={"Time": "date"})
    df["date"] = pd.to_datetime(df["date"])

    # 2) Wide -> long
    df_long = df.melt(
        id_vars="date",
        var_name="ticker",
        value_name="svi"
    )

    # 3) Expand weekly data to daily data
    out = []

    for ticker, g in df_long.groupby("ticker"):
        g = g.sort_values("date").set_index("date")
        
        # Create a daily index from first observed date to 6 days after last observed date
        daily_index = pd.date_range(
            start=g.index.min(),
            end=g.index.max() + pd.Timedelta(days=6),
            freq="D"
        )
        
        # Forward-fill each weekly value across the following days
        g_daily = g.reindex(daily_index).ffill()

        # for the ticker, only keep what comes after ":" in the original ticker name
        ticker = ticker.split(":")[-1]
        g_daily["ticker"] = ticker
        g_daily = g_daily.reset_index().rename(columns={"index": "date"})
        
        out.append(g_daily)

    daily_long = pd.concat(out, ignore_index=True)

    # Keep only the required column order
    daily_long = daily_long[["date", "ticker", "svi"]]
    add_df = pd.concat([add_df, daily_long], ignore_index=True)

add_df.tail(2)



,date,ticker,svi
11128,2021-01-01,UBER,0.0
11129,2021-01-02,UBER,0.0


impute the data set with the new values.

In [32]:
# Make sure dates are datetime in both datasets
add_df["date"] = pd.to_datetime(add_df["date"])

# Rename the imputation source svi to avoid confusion after merge
add_df = add_df.rename(columns={"svi": "svi_imputed_source"})

# Merge
df_final = df_final.merge(
    add_df[["date", "ticker", "svi_imputed_source"]],
    on=["date", "ticker"],
    how="left"
)

# Fill missing svi in main with the value from the daily_svi dataset
df_final["svi"] = df_final["svi"].fillna(df_final["svi_imputed_source"])

# Optional: drop helper column
df_final = df_final.drop(columns="svi_imputed_source")

Now, I look up the number of unique tickers with svi data.

In [37]:
# number of tickers with svi data
valid_tickers = df_final[df_final["svi"].notna()]["ticker"].unique()
print(f"Number of unique tickers with svi data: {len(valid_tickers)}")

Number of unique tickers with svi data: 1707


In [38]:
# new main
new_main = df_final.loc[df_final["ticker"].isin(valid_tickers)].copy()

# build relevant variables for new_main
new_main["users_close_l1"] = new_main.groupby("ticker")["users_close"].shift(1)

# userchg: change in users from day t-1 to day t
new_main["userchg"] = new_main["users_close"] - new_main["users_close_l1"]

# userratio: users_close(t) / users_close(t-1)
new_main["userratio"] = new_main["users_close"] / new_main["users_close_l1"]

# replace inf and -inf in userratio with NaN
new_main["userratio"] = new_main["userratio"].replace([np.inf, -np.inf], np.nan)

# make sure that I take the absolute value of the price
new_main["prc"] = new_main["prc"].abs()

# size is defined as prc * shrout
new_main["size"] = new_main["prc"] * new_main["shrout"] / 1e6  # in millions

# build 'daily_buys'
new_main["daily_buys"] = new_main["buy_num_trades_retail"]

# build 'daily_sells'
new_main["daily_sells"] = new_main["sell_num_trades_retail"]

# build 'net_buys'
new_main["net_buys"] = new_main["daily_buys"] - new_main["daily_sells"]

# build 'taq_retimb'
new_main['taq_retimb'] = new_main['net_buys'] / (new_main['daily_buys'] + new_main['daily_sells'])

# build binary variable 'has_users' indicating whether there are any users for that ticker on that day
new_main["has_users"] = (new_main["users_close"] > 0).astype(int)

Once I computed the relevant variables, I compute summary statistics.

In [39]:
# summary stats Panel A
summary_stats_panel_a = new_main[["users_close", "users_last", "userchg", "userratio", "prc", "size", "ret",
                                "daily_buys", "daily_sells", "net_buys", "taq_retimb"]].describe().T
print(summary_stats_panel_a)

                count         mean           std            min        25%  \
users_close  790317.0  3070.320797  25063.018877       0.000000  70.000000   
users_last   792322.0  3067.486230  25050.082421       0.000000  70.000000   
userchg      783854.0    11.974753    686.623390 -548766.000000  -1.000000   
userratio    781894.0     1.006354      0.335045       0.000000   0.995257   
prc          917224.0    52.990309    145.305788       0.030000  12.130000   
size         917224.0     4.505260     40.661407       0.000092   0.439686   
ret          917224.0     0.000267      0.039282      -1.000000  -0.012226   
daily_buys   909474.0   188.362364   1405.371954       0.000000  16.000000   
daily_sells  909474.0   169.733239   1073.026955       0.000000  16.000000   
net_buys     909474.0    18.629125    491.795439  -30215.000000  -9.000000   
taq_retimb   909474.0    -0.000750      0.265974      -1.000000  -0.126761   

                    50%         75%            max  
users_clos

In [40]:
# Panel B: Daily Observations (Summed Variable Averaged across Days)
# keep only observations where has_users == 1
new_main_agg = new_main[new_main["has_users"] == 1].copy()

# group by date and sum the relevant variables: [users_close, users_last, userchg, daily_buys, daily_sells, net_buys]
daily_summary = new_main_agg.groupby("date").agg({
    "users_close": "sum",
    "users_last": "sum",
    "userchg": "sum",
    "daily_buys": "sum",
    "daily_sells": "sum",
    "net_buys": "sum",
    "has_users": "sum"
}).reset_index()

# summary stats Panel B
summary_stats_panel_b = daily_summary[["users_close", "users_last", "userchg", "daily_buys", "daily_sells", "net_buys", "has_users"]].describe().T
print(summary_stats_panel_b)

             count          mean           std        min        25%  \
users_close  543.0  4.468742e+06  2.571142e+06  1688993.0  2563347.5   
users_last   543.0  4.471471e+06  2.573861e+06  1689464.0  2563917.0   
userchg      543.0  1.729205e+04  3.821022e+04  -472938.0     3420.5   
daily_buys   543.0  2.766435e+05  1.614224e+05    86728.0   176596.5   
daily_sells  543.0  2.483635e+05  1.299733e+05    82086.0   166185.0   
net_buys     543.0  2.828005e+04  3.660912e+04   -15028.0     7221.0   
has_users    543.0  1.451825e+03  2.959494e+01      882.0     1436.0   

                   50%        75%         max  
users_close  3803401.0  4817508.0  10848908.0  
users_last   3804425.0  4821866.0  10858489.0  
userchg         7292.0    16754.5    274769.0  
daily_buys    204907.0   315981.5   1129829.0  
daily_sells   191307.0   280277.5    780362.0  
net_buys       15074.0    32542.0    349467.0  
has_users       1452.0     1466.0      1490.0  


I write `new_main` to csv

In [41]:
# write new main to csv
new_main.to_csv("../../../data/processed/final_sample.csv", index=False)